# 05 · A/B testing and statistics for evals

After this notebook you can answer "is model B really better?" with statistics instead of vibes. That covers bootstrap confidence intervals, paired tests (McNemar and a permutation test, both from scratch and checked against SciPy), how many eval items you need, multiple-comparison and peeking traps, two-proportion z-tests and sample-size planning for online A/B tests, and guardrail checks.

**Time:** ~35 min · **Runs:** offline on CPU in < 30 s (no models, no LLM calls) · **Needs:** [NumPy for AI engineers](../04_numpy_pandas_pytorch/01_numpy_for_ai_engineers.ipynb), [02 · LLM-as-judge](02_llm_as_judge_and_its_biases.ipynb) for where the scores come from

## Why this matters in interviews

- "Model B scores 84% and A scores 81%. Ship B?" is a favourite trap. The right answer starts with "on how many items, paired or not, and what's the interval?"
- A/B testing an LLM feature properly (`ev10`, `in38`, `rg43`) means pre-declared metrics, a power calculation, guardrails and no peeking. Interviewers probe each one.
- "Offline improved, production got worse" (`ev26`) and "is the cheaper model good enough?" (`tl25`) both come down to measurement noise and what you measured.
- Being able to write a bootstrap or a McNemar test in ten lines is a strong practical signal.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ev10` | How do you A/B test an LLM feature properly? |
| `in38` | How would you A/B test two models in production safely? |
| `rg43` | How do you A/B test a retrieval change safely? |
| `ev09` | What is offline versus online evaluation, and why do they disagree? |
| `ev26` | Your offline evals improved and production got worse. What happened? |
| `tl25` | How would you evaluate whether a cheaper model is good enough to switch to? |
| `dt22` | How do you sample data for evaluation without biasing the result? |
| `po29` | What is a canary deployment and what would you watch during one for an LLM feature? |
| `ev28` | How often should evals run, and what gates what? |

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 140)
rng = np.random.default_rng(0)

## 1. Why a single score is not enough

**In plain English:** an eval set is a *sample* of the questions your users could ask. Score the same model on a different sample of 100 questions and you get a different number. So before comparing two scores, ask how much a score moves by chance alone.

Below, a model whose true accuracy is 0.83 is scored on 2,000 different random 100-item eval sets.

In [ ]:
true_acc, n_items = 0.83, 100
scores = rng.binomial(n_items, true_acc, size=2000) / n_items
lo, hi = np.percentile(scores, [2.5, 97.5])
print(f"true accuracy {true_acc}; observed scores on 100-item sets range {scores.min():.2f}-{scores.max():.2f}; "
      f"95% of them fall in [{lo:.2f}, {hi:.2f}]")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(scores, bins=np.arange(0.68, 0.97, 0.01), color="#3b5bdb", edgecolor="white")
ax.axvline(true_acc, color="black", ls="--", label="true accuracy")
ax.axvspan(lo, hi, color="orange", alpha=0.15, label="middle 95% of observed scores")
ax.set_xlabel("observed accuracy on a 100-item eval set"); ax.set_ylabel("count")
ax.set_title("The same model, 2,000 different eval sets: a 3-point gap is inside the noise"); ax.legend(); plt.show()
assert hi - lo > 0.12          # +-7 points of pure sampling noise at n = 100

## 2. Bootstrap confidence intervals

You only have *one* eval set, so resample it. Draw n items **with replacement** from your n results, recompute the accuracy, and repeat thousands of times. The spread of those resampled scores approximates the sampling noise. The middle 95% is a **percentile bootstrap CI**. The bootstrap works for any metric (F1, nDCG, judge scores, p95 latency), which is why it is the default tool in eval work.

We check ours against `scipy.stats.bootstrap` and against the analytic intervals for a proportion: Wald, p ± 1.96·√(p(1−p)/n), and Wilson.

In [ ]:
def bootstrap_ci(values, stat=np.mean, n_boot=5000, alpha=0.05, seed=0):
    """Percentile bootstrap: resample items with replacement, recompute the statistic."""
    r = np.random.default_rng(seed)
    values = np.asarray(values)
    idx = r.integers(0, len(values), size=(n_boot, len(values)))
    boots = stat(values[idx], axis=1)
    return np.percentile(boots, [100 * alpha / 2, 100 * (1 - alpha / 2)]), boots

results = (rng.random(200) < 0.81).astype(int)             # one eval run: 200 items, 1 = correct
acc = results.mean()
(ci_lo, ci_hi), boots = bootstrap_ci(results)
sp = stats.bootstrap((results,), np.mean, n_resamples=5000, method="percentile", rng=np.random.default_rng(1)).confidence_interval
wald = 1.96 * math.sqrt(acc * (1 - acc) / len(results))
wilson = stats.binomtest(int(results.sum()), len(results)).proportion_ci(method="wilson")
print(f"accuracy {acc:.3f}")
print(f"  our bootstrap   [{ci_lo:.3f}, {ci_hi:.3f}]")
print(f"  scipy bootstrap [{sp.low:.3f}, {sp.high:.3f}]")
print(f"  Wald            [{acc - wald:.3f}, {acc + wald:.3f}]")
print(f"  Wilson          [{wilson.low:.3f}, {wilson.high:.3f}]")
assert abs(ci_lo - sp.low) < 0.012 and abs(ci_hi - sp.high) < 0.012         # same method, different random draws
assert abs(ci_lo - (acc - wald)) < 0.015 and abs(ci_hi - (acc + wald)) < 0.015

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(boots, bins=40, color="#3b5bdb", edgecolor="white")
for v in (ci_lo, ci_hi):
    ax.axvline(v, color="orange", lw=2)
ax.axvline(acc, color="black", ls="--")
ax.set_xlabel("bootstrap accuracy"); ax.set_ylabel("count")
ax.set_title(f"5,000 bootstrap resamples: 95% CI [{ci_lo:.3f}, {ci_hi:.3f}]"); plt.show()

## 3. How many eval items do you need?

The half-width of a 95% interval for an accuracy near p is about 1.96·√(p(1−p)/n). It shrinks with **√n**: to halve the noise you need **four times** the items. Plan the eval set size from the smallest difference you care about.

In [ ]:
p = 0.8
ns = np.array([25, 50, 100, 200, 400, 800, 1600])
half = lambda n: 1.96 * np.sqrt(p * (1 - p) / n)
boot_half = [np.diff(bootstrap_ci((np.random.default_rng(int(n)).random(n) < p).astype(int), n_boot=2000)[0])[0] / 2 for n in ns]
assert abs(half(400) - half(100) / 2) < 1e-12                          # 4x the items -> half the width
assert all(abs(b - half(n)) / half(n) < 0.25 for b, n in zip(boot_half, ns) if n >= 100)   # small n: the sample's own accuracy wanders

grid = np.logspace(np.log10(20), np.log10(2000), 100)
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(grid, 100 * half(grid), label="analytic: 1.96·sqrt(p(1-p)/n)")
ax.plot(ns, 100 * np.array(boot_half), "o", label="bootstrap on one simulated eval set")
ax.set_xscale("log"); ax.set_xlabel("eval items (log scale)"); ax.set_ylabel("± percentage points (95% CI)")
ax.set_title("CI half-width vs eval set size (accuracy ≈ 0.8)"); ax.legend(); plt.show()

need = {f"±{m} pts": math.ceil(1.96 ** 2 * p * (1 - p) / (m / 100) ** 2) for m in (10, 5, 3, 2, 1)}
print("items needed for a given 95% half-width at accuracy 0.8:", need)

**Rule of thumb:** 100 items gives about ±8 points and 400 gives about ±4. A 50-case golden set is fine for catching *breakages* (regression tests, assertions). It is far too small to rank two models that differ by 2 points, unless you use a **paired** test, which is next.

## 4. Paired comparisons: same items, two models

If both models answer the **same** items, compare them item by item. Most items are easy or hard for *both*, and that shared difficulty cancels out. Only the **discordant** items carry information: A right and B wrong (b), or A wrong and B right (c). That makes a paired test far more sensitive than comparing two independent accuracies.

- **McNemar's test:** under "no difference", each discordant item is a coin flip. The exact test is a binomial test of b against b + c at 0.5. The classic chi-square version is (|b − c| − 1)² / (b + c) with 1 degree of freedom.
- **Paired permutation test:** randomly swap the two models' results within each item, many times, and see how often the difference in means is as large as observed. It works for any per-item score (judge 1–5, F1, latency). For 0/1 scores it is essentially the exact McNemar test.

In [ ]:
r4, n = np.random.default_rng(2), 400
difficulty = r4.normal(0, 1.2, n)                          # both models find the same items easy or hard
p_a, p_b = 1 / (1 + np.exp(-(1.3 - difficulty))), 1 / (1 + np.exp(-(1.75 - difficulty)))
u = r4.random(n)                                           # shared "luck" per item: results are strongly correlated,
u_a = np.where(r4.random(n) < 0.3, r4.random(n), u)        # as they are for two LLMs on the same questions
u_b = np.where(r4.random(n) < 0.3, r4.random(n), u)
a, b_ = (u_a < p_a).astype(int), (u_b < p_b).astype(int)
table = pd.crosstab(pd.Series(a, name="A correct"), pd.Series(b_, name="B correct"))
b, c = int(((a == 1) & (b_ == 0)).sum()), int(((a == 0) & (b_ == 1)).sum())
print(f"accuracy A = {a.mean():.3f}, B = {b_.mean():.3f}, difference = {b_.mean() - a.mean():+.3f}")
print(table.to_string())
print(f"discordant: A-only b = {b}, B-only c = {c}; items where they agree: {n - b - c}")

In [ ]:
def mcnemar(b, c, exact=True):
    """McNemar test on discordant counts. Exact: binomial; otherwise chi-square with continuity correction."""
    if exact:
        return min(1.0, 2 * stats.binom.cdf(min(b, c), b + c, 0.5))
    stat = (abs(b - c) - 1) ** 2 / (b + c)
    return stats.chi2.sf(stat, df=1)

def two_proportion_z(x1, n1, x2, n2):
    """Unpaired z-test (pooled). Wrong for paired data - shown here for contrast."""
    p1, p2, pool = x1 / n1, x2 / n2, (x1 + x2) / (n1 + n2)
    z = (p2 - p1) / math.sqrt(pool * (1 - pool) * (1 / n1 + 1 / n2))
    return z, 2 * stats.norm.sf(abs(z))

def paired_permutation(x, y, n_perm=20000, seed=0):
    """Two-sided sign-flip test on per-item differences."""
    d = np.asarray(y, float) - np.asarray(x, float)
    signs = np.random.default_rng(seed).choice([-1, 1], size=(n_perm, len(d)))
    null = (signs * d).mean(axis=1)
    return (np.abs(null) >= abs(d.mean()) - 1e-12).mean(), null

p_exact, p_chi2 = mcnemar(b, c), mcnemar(b, c, exact=False)
p_perm, null = paired_permutation(a, b_)
p_unpaired = two_proportion_z(a.sum(), n, b_.sum(), n)[1]
p_scipy_binom = stats.binomtest(min(b, c), b + c, 0.5).pvalue
p_scipy_perm = stats.permutation_test((a, b_), lambda x, y, axis: np.mean(y - x, axis=axis), permutation_type="samples",
                                      n_resamples=20000, rng=np.random.default_rng(3)).pvalue
print(f"McNemar exact      p = {p_exact:.4f}   (scipy binomtest {p_scipy_binom:.4f})")
print(f"McNemar chi-square p = {p_chi2:.4f}")
print(f"paired permutation p = {p_perm:.4f}   (scipy permutation_test {p_scipy_perm:.4f})")
print(f"UNPAIRED z-test    p = {p_unpaired:.4f}   <- ignores the pairing, throws away power")
assert abs(p_exact - p_scipy_binom) < 1e-12
assert abs(p_perm - p_scipy_perm) < 0.01 and abs(p_perm - p_exact) < 0.015      # for 0/1 scores: permutation ~ exact McNemar
assert p_exact < p_unpaired                                                     # pairing buys power

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].imshow(table.values, cmap="Blues")
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, table.values[i, j], ha="center", va="center", fontsize=14,
                     color="white" if table.values[i, j] > table.values.max() / 2 else "black")
axes[0].set_xticks([0, 1], ["B wrong", "B right"]); axes[0].set_yticks([0, 1], ["A wrong", "A right"])
axes[0].set_title("Only the off-diagonal (discordant) cells matter")
axes[1].hist(null, bins=np.arange(null.min() - 1 / n, null.max() + 2 / n, 2 / n), color="#adb5bd", edgecolor="white")
axes[1].axvline(b_.mean() - a.mean(), color="red", lw=2, label="observed difference")
axes[1].axvline(-(b_.mean() - a.mean()), color="red", lw=1, ls="--")
axes[1].set_xlabel("difference in accuracy under 'no real difference'"); axes[1].legend()
axes[1].set_title(f"Permutation null distribution: p = {p_perm:.4f}")
plt.tight_layout(); plt.show()

**Reading it:** the unpaired test treats the two score lists as unrelated samples and wastes the shared difficulty. The paired tests look only at the items where the models disagree. The same data gives a far smaller p-value. **Always evaluate candidate models on the same items, and use a paired test.** For a CI on the *difference*, bootstrap the per-item differences (Try it yourself 1).

## 5. Online A/B tests: the two-proportion z-test, power and sample size

Online, users are split at random into control and treatment, so the groups are independent and the **unpaired** test is the right one. For a binary metric such as "task resolved without escalation":

z = (p₂ − p₁) / √( p̄(1 − p̄)(1/n₁ + 1/n₂) ), where p̄ is the pooled rate. The two-sided p-value is 2·(1 − Φ(|z|)).

It is the same test as a 2×2 chi-square without continuity correction: z² = χ². The assert below checks it.

In [ ]:
x1, n1, x2, n2 = 960, 8000, 1048, 8000                      # control 12.0% resolved, treatment 13.1%
z, p_ab = two_proportion_z(x1, n1, x2, n2)
chi2_stat, p_chi, _, _ = stats.chi2_contingency([[x1, n1 - x1], [x2, n2 - x2]], correction=False)
print(f"control {x1 / n1:.3f}, treatment {x2 / n2:.3f}, lift {x2 / n2 - x1 / n1:+.4f}: z = {z:.3f}, p = {p_ab:.4f}")
assert abs(z ** 2 - chi2_stat) < 1e-9 and abs(p_ab - p_chi) < 1e-12

diff = x2 / n2 - x1 / n1
se_unpooled = math.sqrt(x1 / n1 * (1 - x1 / n1) / n1 + x2 / n2 * (1 - x2 / n2) / n2)
print(f"95% CI for the lift: [{diff - 1.96 * se_unpooled:+.4f}, {diff + 1.96 * se_unpooled:+.4f}]")

**Plan the sample size before launch.** To detect a lift δ from a baseline rate p₁ with significance α (two-sided) and power 1 − β:

n per arm = (z₁₋α/₂ + z₁₋β)² · (p₁(1−p₁) + p₂(1−p₂)) / δ²

For a 12% baseline, a +1-point minimum detectable effect, α = 0.05 and 80% power, that is roughly **17,000 users per arm**. The cell checks the formula by simulating thousands of experiments at that size and counting how often the test fires.

In [ ]:
def n_per_arm(p1, mde, alpha=0.05, power=0.8):
    p2 = p1 + mde
    za, zb = stats.norm.ppf(1 - alpha / 2), stats.norm.ppf(power)
    return math.ceil((za + zb) ** 2 * (p1 * (1 - p1) + p2 * (1 - p2)) / mde ** 2)

def simulated_power(p1, p2, n, sims=4000, seed=0):
    r = np.random.default_rng(seed)
    c1, c2 = r.binomial(n, p1, sims), r.binomial(n, p2, sims)
    pool = (c1 + c2) / (2 * n)
    z = (c2 - c1) / n / np.sqrt(pool * (1 - pool) * 2 / n)
    return (np.abs(z) > stats.norm.ppf(0.975)).mean()

n_req = n_per_arm(0.12, 0.01)
pw = simulated_power(0.12, 0.13, n_req)
print(f"n per arm for 12% -> 13% at 80% power: {n_req:,}; simulated power at that n: {pw:.3f}")
assert 16_000 < n_req < 18_500 and abs(pw - 0.80) < 0.03

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
mdes = np.linspace(0.005, 0.05, 40)
for base in (0.05, 0.12, 0.30):
    axes[0].plot(100 * mdes, [n_per_arm(base, m) for m in mdes], label=f"baseline {base:.0%}")
axes[0].set_yscale("log"); axes[0].set_xlabel("minimum detectable effect (percentage points)")
axes[0].set_ylabel("users per arm (log)"); axes[0].set_title("Halving the MDE quadruples the traffic"); axes[0].legend()
n_grid = np.array([2000, 5000, 10000, 17000, 25000, 40000])
axes[1].plot(n_grid, [simulated_power(0.12, 0.13, int(m)) for m in n_grid], "o-", label="simulated power")
axes[1].axhline(0.8, ls="--", color="gray"); axes[1].axvline(n_req, ls=":", color="gray")
axes[1].set_xlabel("users per arm"); axes[1].set_ylabel("power"); axes[1].set_title("Power for 12% -> 13%")
axes[1].legend(); plt.tight_layout(); plt.show()

## 6. Two ways to fool yourself: many variants, and peeking

**Multiple comparisons.** Test 20 prompt variants against a baseline at α = 0.05 when *none* of them is better, and the chance that at least one looks "significant" is 1 − 0.95²⁰ ≈ **64%**. Fixes:
- **Bonferroni:** use α/m. It is simple and conservative.
- **Holm:** step down through the sorted p-values. It has the same guarantee and more power.
- **Benjamini–Hochberg:** controls the *false discovery rate* instead. It suits screening many variants.
- Or confirm the winner on a **fresh** held-out set (`ev15`).

**Peeking.** Checking an A/B test every day and stopping the first time p < 0.05 inflates false positives in the same way, because each look is another comparison (`ev10`: "you will peek your way into a false positive"). Fix it by fixing the sample size in advance, or by using a sequential method designed for repeated looks (alpha spending, mSPRT, always-valid p-values).

Both are simulated below on **A/A tests**, where there is no real difference, so every "win" is a false positive. The 64% assumes independent tests. When all 20 variants are compared with the *same* baseline run, the tests are correlated (an unlucky baseline makes many variants look good at once). The rate comes out lower, but still many times 5%.

In [ ]:
def z_p(c1, c2, n):                              # vectorised two-proportion z-test p-values, equal arms
    pool = (c1 + c2) / (2 * n)
    z = (c2 - c1) / n / np.sqrt(np.maximum(pool * (1 - pool), 1e-12) * 2 / n)
    return 2 * stats.norm.sf(np.abs(z))

r = np.random.default_rng(5)
sims, m, n_eval = 3000, 20, 400
variants = r.binomial(n_eval, 0.8, (sims, m))                          # 20 variants, all truly equal to the baseline
pvals = z_p(r.binomial(n_eval, 0.8, (sims, m)), variants, n_eval)      # (a) each test with its own baseline run
pvals_shared = z_p(r.binomial(n_eval, 0.8, sims)[:, None], variants, n_eval)   # (b) all 20 vs ONE baseline run
fwer = lambda k, thr, pv=pvals: (pv[:, :k] < thr).any(axis=1).mean()
print(f"P(at least one false 'win' among 20): independent tests {fwer(20, 0.05):.3f} (theory {1 - 0.95 ** 20:.3f}); "
      f"shared baseline {fwer(20, 0.05, pvals_shared):.3f}")
print(f"with Bonferroni (0.05/20): {fwer(20, 0.05 / 20):.3f} / {fwer(20, 0.05 / 20, pvals_shared):.3f}")
assert abs(fwer(20, 0.05) - (1 - 0.95 ** 20)) < 0.06
assert 0.2 < fwer(20, 0.05, pvals_shared) < fwer(20, 0.05)       # correlated tests: lower, but still far above 5%
assert fwer(20, 0.05 / 20) < 0.08 and fwer(20, 0.05 / 20, pvals_shared) < 0.08

n_total, looks = 10_000, 20                                           # users per arm, and daily looks
step = n_total // looks
ca = r.binomial(step, 0.12, (2000, looks)).cumsum(axis=1)              # cumulative successes at each look (A/A)
cb = r.binomial(step, 0.12, (2000, looks)).cumsum(axis=1)
checkpoints = step * np.arange(1, looks + 1)
p_at = z_p(ca, cb, checkpoints)
fpr_by_looks = [(p_at[:, np.linspace(0, looks - 1, L).astype(int)] < 0.05).any(axis=1).mean() for L in range(1, looks + 1)]
print(f"A/A false-positive rate: 1 look {fpr_by_looks[0]:.3f}, {looks} looks {fpr_by_looks[-1]:.3f}")
assert abs(fpr_by_looks[0] - 0.05) < 0.02 and fpr_by_looks[-1] > 0.12

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
ks = np.arange(1, 31)
axes[0].plot(ks, 1 - 0.95 ** ks, label="theory: 1 - 0.95^m")
axes[0].plot(range(1, m + 1), [fwer(k, 0.05) for k in range(1, m + 1)], "o", ms=4, label="simulated, independent")
axes[0].plot(range(1, m + 1), [fwer(k, 0.05, pvals_shared) for k in range(1, m + 1)], "^", ms=4, label="simulated, shared baseline")
axes[0].plot(range(1, m + 1), [fwer(k, 0.05 / k) for k in range(1, m + 1)], "s", ms=4, label="simulated, Bonferroni")
axes[0].set_xlabel("variants tested"); axes[0].set_ylabel("P(any false win)"); axes[0].legend(); axes[0].set_title("Multiple comparisons")
axes[1].plot(range(1, looks + 1), fpr_by_looks, "o-", color="crimson")
axes[1].axhline(0.05, ls="--", color="gray"); axes[1].set_xlabel("number of peeks (stop at first p < 0.05)")
axes[1].set_ylabel("false-positive rate"); axes[1].set_title("Peeking at an A/A test")
plt.tight_layout(); plt.show()

## 7. Guardrail metrics and sanity checks

A treatment can win the primary metric and still be unshippable. **Guardrails** are pre-declared metrics that must *not get worse* beyond a margin (`ev10`, `po29`): p95 latency, cost per request, error rate, escalation rate, safety incidents, refusal rate. Test them as **non-inferiority**: "the upper end of the CI for the p95-latency ratio is below 1.10". Failing to find a significant regression is not the same test.

Before trusting any A/B result, run a **sample ratio mismatch (SRM)** check. If you meant 50/50 and got 10,000 vs 10,500 (a 49/51 split that looks harmless), randomisation or logging is broken, and every metric is suspect.

In [ ]:
lat_a = rng.lognormal(mean=np.log(1.2), sigma=0.45, size=4000)          # seconds, control
lat_b = rng.lognormal(mean=np.log(1.2) + 0.05, sigma=0.45, size=4000)   # treatment: ~5% slower
def p95_ratio(x, y):
    return np.percentile(y, 95) / np.percentile(x, 95)
rb = np.random.default_rng(9)
boot = np.array([p95_ratio(lat_a[rb.integers(0, len(lat_a), len(lat_a))], lat_b[rb.integers(0, len(lat_b), len(lat_b))])
                 for _ in range(2000)])
lo_r, hi_r = np.percentile(boot, [2.5, 97.5])
margin = 1.10
print(f"p95 latency: control {np.percentile(lat_a, 95):.2f}s, treatment {np.percentile(lat_b, 95):.2f}s, "
      f"ratio {p95_ratio(lat_a, lat_b):.3f}, 95% CI [{lo_r:.3f}, {hi_r:.3f}] -> guardrail (< {margin}) "
      f"{'PASS' if hi_r < margin else 'FAIL'}")

def srm_check(observed, expected_share=(0.5, 0.5)):
    """Chi-square goodness of fit of the traffic split, from scratch."""
    total = sum(observed)
    exp = [total * s for s in expected_share]
    chi2 = sum((o - e) ** 2 / e for o, e in zip(observed, exp))
    return chi2, stats.chi2.sf(chi2, df=len(observed) - 1)

chi2_srm, p_srm = srm_check([10_000, 10_500])
assert abs(p_srm - stats.chisquare([10_000, 10_500]).pvalue) < 1e-12 and p_srm < 0.001
print(f"SRM check 10,000 vs 10,500 (usual alarm: p < 0.001): chi2 = {chi2_srm:.2f}, p = {p_srm:.4f} -> {'investigate before reading any metric' if p_srm < 0.001 else 'ok'}")

fig, ax = plt.subplots(figsize=(8, 3.6))
bins = np.linspace(0, 5, 60)
ax.hist(lat_a, bins=bins, alpha=0.6, label="control"); ax.hist(lat_b, bins=bins, alpha=0.6, label="treatment")
for x, col in [(np.percentile(lat_a, 95), "C0"), (np.percentile(lat_b, 95), "C1")]:
    ax.axvline(x, color=col, ls="--")
ax.set_xlabel("latency (s)"); ax.set_ylabel("requests"); ax.legend()
ax.set_title("Guardrail: compare the tail (p95, dashed), not the mean"); plt.show()

## 8. Offline versus online evaluation

| | Offline eval (golden set, judge, CI) | Online eval (A/B, canary, shadow) |
|---|---|---|
| Measures | what you thought to include | what users actually do |
| Speed and cost | minutes, cheap, reproducible | days to weeks, needs traffic |
| Statistics | paired tests on the same items | unpaired tests on random user splits |
| Typical metrics | accuracy, faithfulness, judge score, recall@k | resolution, escalation, rephrase rate, retention, p95 latency, cost |
| Fails when | the eval set does not match traffic; overfitting the set | underpowered, peeking, novelty effects, SRM |
| Gates | merges and deploys (`ev28`) | ramps: 1% → 5% → 25% → 50% with automatic rollback |

When they disagree, **online wins**. The useful work is explaining the gap (`ev09`, `ev26`): sample 50 production failures and check how many resemble anything in your eval set. Add what is missing, and stratify the offline set by intent and language (`dt22`). **Shadow deployment** (run the new version on mirrored traffic without serving it) sits in between. It gives real inputs, latency and cost with no user risk, as long as tools with side effects are sandboxed.

## 9. The decision checklist: "is model B really better?"

The function below runs the offline half of the checklist on the paired results from section 4. The online half runs in the A/B test.

In [ ]:
def is_b_better(a, b, min_effect=0.02, alpha=0.05, n_comparisons=1, guardrails_ok=True):
    d = np.asarray(b) - np.asarray(a)
    (lo, hi), _ = bootstrap_ci(d, n_boot=5000, seed=11)
    bb, cc = int((d < 0).sum()), int((d > 0).sum())
    p = mcnemar(bb, cc)
    checks = {
        "same items for both models (paired)": len(a) == len(b),
        f"95% CI of the difference excludes 0 [{lo:+.3f}, {hi:+.3f}]": lo > 0,
        f"p < alpha / comparisons ({p:.4f} < {alpha / n_comparisons:.4f})": p < alpha / n_comparisons,
        f"effect >= practical threshold ({d.mean():+.3f} >= {min_effect})": d.mean() >= min_effect,
        "guardrails (latency, cost, safety) not worse": guardrails_ok,
    }
    for name, ok in checks.items():
        print(f"  [{'x' if ok else ' '}] {name}")
    return all(checks.values())

print("B vs A, one planned comparison:")
print("  -> ship B?", is_b_better(a, b_))
print("\nB vs A, but B was the best of 20 prompt variants tried on this set:")
print("  -> ship B?", is_b_better(a, b_, n_comparisons=20), "(confirm on a fresh held-out set first)")

**The rest of the checklist, where code cannot decide for you:**
1. Was the eval set frozen, versioned and **not** used for tuning B? Confirm on a held-out slice (`ev15`).
2. Is the judge pinned (model version and prompt), and was it the same for A and B (`ev34`)?
3. Does it hold per **segment**: intent, language, customer tier? An average can hide a collapse (`ev10`).
4. Is the run-to-run noise understood? At temperature > 0, re-run and include that variance.
5. Online: primary and guardrail metrics declared up front, a power calculation, randomisation by user, an SRM check, at least a full week, no peeking, and automatic rollback on guardrail breach (`in38`).

## Try it yourself

**1. A CI for the difference.** Bootstrap the per-item differences `b_ - a` from section 4 to get a 95% CI for "B minus A". Compare its width with what you get from bootstrapping each model separately and subtracting, which ignores the pairing.

In [ ]:
# Solution — try it yourself first, then run this
(lo_p, hi_p), _ = bootstrap_ci(b_ - a, seed=1)
r1 = np.random.default_rng(2)
unpaired = [b_[r1.integers(0, n, n)].mean() - a[r1.integers(0, n, n)].mean() for _ in range(5000)]
lo_u, hi_u = np.percentile(unpaired, [2.5, 97.5])
print(f"paired CI   [{lo_p:+.3f}, {hi_p:+.3f}]  width {hi_p - lo_p:.3f}")
print(f"unpaired CI [{lo_u:+.3f}, {hi_u:+.3f}]  width {hi_u - lo_u:.3f}   <- wider: the shared difficulty is not cancelled")

**2. Run-to-run noise.** At temperature > 0, the *same* model scores differently on the *same* items. Simulate it: each item has the per-item success probability `p_b` from section 4. Run model B 200 times on the same 300 items and look at the spread of its accuracy. Is that noise small next to the 3-point A/B difference?

In [ ]:
# Solution — try it yourself first, then run this
reruns = (np.random.default_rng(4).random((200, n)) < p_b).mean(axis=1)
print(f"same model, same items, 200 reruns: accuracy {reruns.mean():.3f} ± {reruns.std():.3f} (sd); "
      f"range {reruns.min():.3f}-{reruns.max():.3f}")
# A 3-point difference is only a few SDs of rerun noise: report the mean over several runs, or use temperature 0.

**3. Holm versus Bonferroni.** Given the p-values from testing 6 variants, [0.001, 0.008, 0.012, 0.03, 0.04, 0.2], which variants are significant at a family-wise α = 0.05 under Bonferroni, and which under Holm?

In [ ]:
# Solution — try it yourself first, then run this
pv = np.array([0.001, 0.008, 0.012, 0.03, 0.04, 0.2]); m_ = len(pv)
bonferroni = pv < 0.05 / m_
order = np.argsort(pv); holm = np.zeros(m_, bool)
for rank, i in enumerate(order):                    # step down: compare the k-th smallest with alpha / (m - k)
    if pv[i] < 0.05 / (m_ - rank):
        holm[i] = True
    else:
        break
print("Bonferroni:", bonferroni.astype(int), " Holm:", holm.astype(int))
assert holm.sum() >= bonferroni.sum()               # Holm never rejects fewer

## Say it in an interview

**30-second answer (`ev10`):** "Before the test I declare the primary metric, say task resolution, and the guardrails: p95 latency, cost per request, escalation and safety incidents. I compute the sample size for the smallest lift I care about, randomise by user, and check the traffic split for SRM. I ramp 1 → 5 → 25 → 50% with automatic rollback on a guardrail breach, run at least a full week without peeking (or use a sequential test), and segment the result by language and intent. Offline, before any of that, I compare the candidates on the same frozen items with a paired test and a bootstrap CI on the difference."

**Numbers to have ready:**
- 95% CI half-width ≈ 1.96·√(p(1−p)/n): about ±8 points at n = 100, ±4 at n = 400. Four times the items halves the noise.
- 20 comparisons at α = 0.05 → a 64% chance of a false win. Bonferroni uses α/m.
- n per arm ≈ 16·p(1−p)/δ² for 80% power at α = 0.05. For 12% → 13%, that is about 17k users per arm.
- McNemar uses only the discordant pairs: exact = binomial(b, b + c, 0.5).

**Follow-ups:**
- *Paired or unpaired?* Paired offline (same items). Unpaired online (random users).
- *Why not just average the thumbs-up rate?* It is a biased sample of the unhappy (`ev11`), useful for finding problems, not for measuring them.
- *The test is significant but the lift is 0.1 points.* Statistical significance is not practical significance. Compare against the minimum effect worth the cost.
- *Novelty effect?* Run long enough to see it decay, and look at the lift by week.

**Traps:**
- Comparing two models on different item sets.
- Picking the best of many variants on the same set and reporting its p-value unadjusted.
- Stopping when p dips below 0.05.
- Guardrails checked as "no significant regression" instead of non-inferiority.
- Ignoring an SRM.

## Next

- [01 · LLM eval metrics](01_llm_eval_metrics_em_f1_bleu_rouge.ipynb) · [02 · LLM-as-judge](02_llm_as_judge_and_its_biases.ipynb) · [03 · RAG evaluation](03_rag_evaluation_retrieval_and_faithfulness.ipynb) · [04 · Agent evaluation](04_agent_evaluation.ipynb)
- [Latency percentiles p50/p95 and cost](../16_production/05_latency_percentiles_p50_p95_and_cost.ipynb) for the guardrail side
- [Metrics, confusion matrix, ROC/PR](../05_machine_learning/04_metrics_confusion_matrix_roc_pr.ipynb)
- [Eval harness from scratch](../17_interview_coding_rounds/05_eval_harness_from_scratch.ipynb)
- Theory: [AI system design concepts](../../ai_system_design_concepts/index.html) · [interview bank](../../ai_interview_prep/index.html)